In [8]:
"""
读写大文件
? read 能否记录上次读的位置
"""
def read_file(file_path):
    BLOCK_SIZE = 100
    with open(file_path, "rb") as f:
        while True:
            block = f.read(BLOCK_SIZE)
            if block:
                yield block
            else:
                return


read_gen = read_file("a.txt")
for data in read_gen:
    print(data)

b'11111\r\n22222\r\n33333\r\n44444'


In [7]:
"""
fab_1 不知用yield
fab_2 使用yield
"""
def fab_1(max):
    n, a, b = 0, 0, 1
    while n < max:
        print(b)
        a, b = b, a + b
        n = n + 1


def fab_2(max):
    n, a, b = 0, 0, 1
    while n < max:
        yield b
        a, b = b, a + b
        n = n + 1


fab_1(5)
print('---------')
for n in fab_2(5):
    print(n)

1
1
2
3
5
---------
1
1
2
3
5


In [13]:
# 子生成器
def average_gen():
    total = 0
    count = 0
    average = 0
    while True:
        new_num = yield average
        if new_num is None:
            break
        count += 1
        total += new_num
        average = total / count
    return total, count, average


# 委托生成器
def proxy_gen():
    while True:
        # 只有子生成器要结束（return）了，yield from左边的变量才会被赋值，后面的代码才会执行。
        total, count, average = yield from average_gen()
        print("计算完毕！！\n总共传入 {} 个数值， 总和：{}，平均数：{}".format(count, total, average))


# 调用方
def main():
    calc_average = proxy_gen()
    next(calc_average)
    print(calc_average.send(10))
    print(calc_average.send(20))
    print(calc_average.send(30))
    calc_average.send(None)
    print(calc_average.send(40))


main()

10.0
15.0
20.0
计算完毕！！
总共传入 3 个数值， 总和：60，平均数：20.0
40.0


In [14]:
# 子生成器
def average_gen():
    total = 0
    count = 0
    average = 0
    while True:
        new_num = yield average
        if new_num is None:
            break
        count += 1
        total += new_num
        average = total / count
    return total, count, average


# 调用方
def main():
    calc_average = average_gen()
    next(calc_average)  # 预激协程
    print(calc_average.send(10))  # 打印：10.0
    print(calc_average.send(20))  # 打印：15.0
    print(calc_average.send(30))  # 打印：20.0
    try:
        calc_average.send(None)
    except StopIteration as e:
        print('error')
        total, count, average = e.value
        print("计算完毕！！\n总共传入 {} 个数值， 总和：{}，平均数：{}".format(count, total, average))


main()

10.0
15.0
20.0
error
计算完毕！！
总共传入 3 个数值， 总和：60，平均数：20.0


In [ ]:
"""
yield from 可以处理异常 

_i：子生成器，同时也是一个迭代器
_y：子生成器生产的值
_r：yield from 表达式最终的值
_s：调用方通过send()发送的值
_e：异常对象
"""

_i = iter(EXPR)

try:
    _y = next(_i)
except StopIteration as _e:
    _r = _e.value

else:
    while 1:
        try:
            _s = yield _y
        except GeneratorExit as _e:
            try:
                _m = _i.close
            except AttributeError:
                pass
            else:
                _m()
            raise _e
        except BaseException as _e:
            _x = sys.exc_info()
            try:
                _m = _i.throw
            except AttributeError:
                raise _e
            else:
                try:
                    _y = _m(*_x)
                except StopIteration as _e:
                    _r = _e.value
                    break
        else:
            try:
                if _s is None:
                    _y = next(_i)
                else:
                    _y = _i.send(_s)
            except StopIteration as _e:
                _r = _e.value
                break
RESULT = _r

In [11]:
"""
yield from 示例
? 生成器第二次yield后为什么没了
"""
astr = 'ABC'
alist = [1, 2, 3]
adict = {"name": "wangbm", "age": 18}
agen = (i for i in range(4, 8))


def gen_1(*args, **kw):
    for item in args:
        yield from item


def gen_2(*args, **kw):
    for item in args:
        for i in item:
            yield i


list_1 = gen_1(astr, alist, adict, agen)
print(list(list_1))
print(list(list_1))
list_2 = gen_2(astr, alist, adict, agen)
print(list(list_2))
print(list(list_2))

['A', 'B', 'C', 1, 2, 3, 'name', 'age', 4, 5, 6, 7]
[]
['A', 'B', 'C', 1, 2, 3, 'name', 'age']
[]
